In [3]:
from pathlib import Path
import sqlite3
from difflib import get_close_matches

# Resolve DB path relative to this notebook location.
db_path = Path.cwd().parent / "SQL" / "IFCAllData.db"
print(f"Using database: {db_path}")

if not db_path.exists():
    raise FileNotFoundError(f"Database not found: {db_path}")

conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Find table names with COBie in the name.
tables = [
    row[0]
    for row in cur.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()
 ]
cobie_candidates = [t for t in tables if "cobie" in t.lower()]

print("COBie-like tables found:")
for t in cobie_candidates:
    print(" -", t)

# Prefer exact known name if present; otherwise take closest match.
preferred_name = "IFCAllData-COBie"
if preferred_name in tables:
    table_name = preferred_name
elif cobie_candidates:
    exactish = get_close_matches(preferred_name, cobie_candidates, n=1)
    table_name = exactish[0] if exactish else cobie_candidates[0]
else:
    raise ValueError("No COBie-related table found in IFCAllData.db")

print(f"\nSelected table: {table_name}")

Using database: c:\Git\APS-IFC\SQL\IFCAllData.db
COBie-like tables found:
 - IFCAllData-COBie

Selected table: IFCAllData-COBie


In [4]:
# Print a sample of the selected COBie table.
preview_sql = f'SELECT * FROM "{table_name}" LIMIT 20'
rows = cur.execute(preview_sql).fetchall()
column_names = [d[0] for d in cur.description]

print(f"Sample rows from {table_name} (up to 20 rows):")
print(" | ".join(column_names))
print("-" * 120)
for r in rows:
    print(" | ".join("" if v is None else str(v) for v in r))

# Count totals.
total_rows = cur.execute(
    f'SELECT COUNT(*) FROM "{table_name}"'
).fetchone()[0]
total_columns = len(column_names)
total_cells = total_rows * total_columns

# Count rows with at least one non-empty value (excluding source file to avoid trivial non-empty rows).
data_columns = [c for c in column_names if c.upper() != "SOURCE_FILE"]
if data_columns:
    any_data_filter = " OR ".join(
        [f'("{c}" IS NOT NULL AND TRIM(CAST("{c}" AS TEXT)) <> "")' for c in data_columns]
    )
    rows_with_any_data = cur.execute(
        f'SELECT COUNT(*) FROM "{table_name}" WHERE {any_data_filter}'
    ).fetchone()[0]
else:
    rows_with_any_data = 0

# Count non-empty cells specifically in COBie column.
if "COBie" in column_names:
    cobie_data_cells = cur.execute(
        f'''SELECT COUNT(*)
        FROM "{table_name}"
        WHERE "COBie" IS NOT NULL
          AND TRIM(CAST("COBie" AS TEXT)) <> '' '''
    ).fetchone()[0]
else:
    cobie_data_cells = 0

print("\nCounts:")
print(f"- Total rows in table: {total_rows}")
print(f"- Total columns in table: {total_columns}")
print(f"- Total cells in table (rows x columns): {total_cells}")
print(f"- Rows containing COBie-related data (excluding SOURCE_FILE-only rows): {rows_with_any_data}")
print(f"- Cells with value in COBie column: {cobie_data_cells}")

conn.close()

Sample rows from IFCAllData-COBie (up to 20 rows):
SOURCE_FILE | NAME | GLOBALID | COBie
------------------------------------------------------------------------------------------------------------------------
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 
ACD-18040-ALL-ST-N2x3.json |  |  | 

Counts:
- Total rows in table: 31241
- Total columns in table: 4
- To